### Aula 01 - Fundamentos do Apache Spark

#### Objetivos

- Entender o que é Spark
- Criar uma SparkSession
- Ler arquivos CSV
- Conhecer DataFrames
- Entender Lazy Evaluation
- Diferenciar Transformations e Actions
- Introdução a Column
- Schema e inferSchema


#### 1. Inicialização da aplicação Spark

O Apache Spark processa dados por meio de tarefas que trabalham sobre partições. A API de DataFrames permite descrever esse processamento em termos de colunas e operações, enquanto o motor planeja como executá-lo. Este primeiro contato concentra-se na sessão, na leitura e na inspeção dos dados; não há medição de paralelismo ou desempenho neste notebook.

`SparkSession` é o ponto de entrada para a API estruturada. As duas células seguintes importam a classe e obtêm uma sessão com o nome `aula_01_fundamentos`. `getOrCreate()` pode reutilizar uma sessão existente; o nome da aplicação ajuda a identificá-la nas ferramentas de acompanhamento, sem definir sozinho quantos recursos serão usados.


In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = (
    SparkSession.builder
    .appName("aula_01_fundamentos")
    .getOrCreate()
)

#### 2. Ingestão do CSV e primeira action

A leitura usa o arquivo de clientes da Aula 01. `header=True` interpreta a primeira linha como nomes de campos; `inferSchema=True` identifica tipos a partir dos valores, pois o CSV não armazena um schema tipado. A inferência exige uma passagem adicional pelos dados e pode realizar trabalho já durante a criação do DataFrame. Assim, lazy evaluation não significa que toda chamada de leitura seja isenta de acesso à fonte. [Referência: leitura de CSV no Spark](https://spark.apache.org/docs/3.5.8/sql-data-sources-csv.html).

O DataFrame `df` associa um schema a um plano de consulta, em vez de carregar a tabela inteira como uma coleção Python. `show()` é uma **action**: solicita registros concretos para exibição e dispara o processamento necessário. A saída salva apresenta 20 clientes e informa que há mais linhas; ela não fornece a contagem total nem garante uma ordem sem `orderBy()`.

Na tabela observada, `idade` e `score` contêm inteiros, `renda_mensal` contém valores numéricos e `ativo` aparece como booleano. Essa inspeção inicial ajuda a conferir se a ingestão corresponde ao contrato de dados esperado antes de construir um pipeline.


In [ ]:
df = spark.read.csv("/workspace/datasets/01_fundamentos/clientes.csv", header= True, inferSchema= True)
df.show()# Action

+----------+------------+-----+--------------+---+------------+-----+-----+
|cliente_id|        nome|idade|        cidade| uf|renda_mensal|ativo|score|
+----------+------------+-----+--------------+---+------------+-----+-----+
|         1|Cliente_0001|   25|Belo Horizonte| MG|     1912.68| true|  550|
|         2|Cliente_0002|   26|        Recife| PE|    13651.78| true|  904|
|         3|Cliente_0003|   20|     São Paulo| SP|     1991.65| true|  538|
|         4|Cliente_0004|   56|Rio de Janeiro| RJ|     1937.84| true|  965|
|         5|Cliente_0005|   52|Belo Horizonte| MG|     8422.08|false|  903|
|         6|Cliente_0006|   69|     Fortaleza| CE|    15843.46| true|  732|
|         7|Cliente_0007|   35|     Fortaleza| CE|     4065.41| true|  404|
|         8|Cliente_0008|   42|      Salvador| BA|     3095.82| true|  918|
|         9|Cliente_0009|   69|     Fortaleza| CE|     2216.93|false|  849|
|        10|Cliente_0010|   42|      Salvador| BA|      2800.2| true|  943|
|        11|

#### 3. Inspeção do plano físico de leitura

A próxima célula chama `explain()` sem o modo estendido: **a saída salva contém apenas o Physical Plan**. O comentário original menciona planos físico/lógico, mas, nesta chamada, o relatório lógico não é exibido. O método descreve o planejamento da consulta; não apresenta as linhas nem comprova uma execução completa. [Referência: `DataFrame.explain`](https://spark.apache.org/docs/3.5.8/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html).

O operador observado, `FileScan csv`, identifica a fonte de leitura. `Location` aponta para o arquivo e `ReadSchema` descreve os campos solicitados ao leitor; a impressão desse schema está abreviada na saída. `DataFilters`, `PushedFilters` e `PartitionFilters` estão vazios porque a consulta inspecionada ainda não aplica filtros. `Batched: false` informa que esse leitor não está produzindo lotes colunares nesse plano.

O Spark usa informações de schema e expressões para analisar e otimizar consultas com o Catalyst. Aqui, como há apenas leitura, não aparecem operações adicionais de projeção ou filtragem. Inspecionar o plano permite relacionar o que foi pedido pela API à estratégia escolhida, sem presumir ganhos de desempenho.


In [ ]:
df.explain() # Exibe o plano físico/lógico de execução


== Physical Plan ==
FileScan csv [cliente_id#92,nome#93,idade#94,cidade#95,uf#96,renda_mensal#97,ativo#98,score#99] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/01_fundamentos/clientes.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,nome:string,idade:int,cidade:string,uf:string,renda_mensal:double,ativo:boo...




#### 4. Representação do DataFrame e schema

`print(df)` exibe uma representação textual do objeto com nomes e tipos das colunas; não solicita registros como `show()`. A saída salva confirma, entre outros campos, `cliente_id: int`, `renda_mensal: double` e `ativo: boolean`.

Essa representação é útil para uma inspeção rápida do contrato do DataFrame, mas não substitui a verificação dos valores. Um schema numérico, por exemplo, não garante por si só que toda renda seja válida para a regra de negócio. A eventual leitura realizada antes pela inferência de tipos não é um efeito dessa chamada a `print()`.


In [ ]:
print(df)# Representação do DataFrame (não executa leitura dos dados)

DataFrame[cliente_id: int, nome: string, idade: int, cidade: string, uf: string, renda_mensal: double, ativo: boolean, score: int]


#### 5. DataFrame e Column: objetos com papéis diferentes

As duas impressões seguintes confirmam as classes `pyspark.sql.dataframe.DataFrame` e `pyspark.sql.column.Column`. `df` representa a relação distribuída; `df.idade` representa uma **expressão de coluna**, que poderá ser usada em filtros, projeções e cálculos. Acessar essa expressão não extrai todos os valores de idade para uma lista Python.

Nas próximas aulas, transformações construirão novos DataFrames a partir de expressões como essa. A imutabilidade da API significa que essas operações devolvem novos planos, preservando o DataFrame de origem; reatribuir uma variável muda a referência Python, não edita o CSV.


In [ ]:
print(type(df))# DataFrame x Column
print(type(df.idade))

<class 'pyspark.sql.dataframe.DataFrame'>
<class 'pyspark.sql.column.Column'>


#### 6. Encerramento da sessão

`spark.stop()` encerra o contexto Spark associado à sessão e libera seus recursos. Essa chamada conclui o ciclo de vida da aplicação; não transforma os clientes nem grava dados. O resumo original abaixo consolida os conceitos apresentados.


In [14]:
spark.stop()

#### Resumo

Nesta aula aprendemos que:

- DataFrames são imutáveis.
- Transformations não executam imediatamente.
- Actions disparam a execução.
- Column representa uma expressão.
- Spark constrói um plano antes de executar.